# 3/11 test 시작

In [2]:
import os, cv2, pandas as pd, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models.video import swin3d_t, Swin3D_T_Weights
from tqdm import tqdm
import gc

# NumPy 2.0 호환성 패치
if not hasattr(np, 'complex_'): np.complex_ = np.complex128
if not hasattr(np, 'float_'): np.float_ = np.float64
if not hasattr(np, 'int_'): np.int_ = np.int64

# 경로 및 하드웨어 설정
class DAiSEEConfig:
    ROOT = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE"
    VIDEO = os.path.join(ROOT, "DataSet")
    LABEL = os.path.join(ROOT, "Labels")
    TRAIN_CSV = os.path.join(LABEL, "AllLabels.csv")
    SAVE_PATH = "stable_daisee_model.pth"
    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    BATCH_SIZE = 2
    NUM_WORKERS = 0 
    LR = 1e-4
    WEIGHTS = torch.tensor([5.0, 3.5, 2.0, 0.6]).to(DEVICE)

# 데이터셋 클래스
class DAiSEEDataset(Dataset):
    def __init__(self, csv, root, transform=None):
        if not os.path.exists(csv):
            raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {csv}")
        self.data, self.root, self.transform = pd.read_csv(csv), root, transform

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        try:
            name = str(self.data.iloc[idx, 0])
            v_path = os.path.join(self.root, name if name.endswith('.mp4') else name + '.mp4')
            
            if not os.path.exists(v_path):
                return torch.zeros(16, 3, 224, 224), 0

            cap = cv2.VideoCapture(v_path)
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            frames = []
            
            if total >= 16:
                indices = np.linspace(0, total - 1, 16).astype(int)
                for f_idx in indices:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
                    ret, f = cap.read()
                    if ret:
                        f = cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (224, 224))
                        if self.transform: f = self.transform(f)
                        frames.append(f)
                    else: break
            cap.release()
            
            while len(frames) < 16:
                frames.append(torch.zeros(3, 224, 224) if not frames else frames[-1])
            return torch.stack(frames), int(self.data.iloc[idx, 2])
        except Exception:
            return torch.zeros(16, 3, 224, 224), 0

# 모델 정의
class DAiSEEModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.swin = swin3d_t(weights=Swin3D_T_Weights.DEFAULT)
        self.swin.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(self.swin.head.in_features, 4))
    def forward(self, x): return self.swin(x.transpose(1, 2))

if __name__ == "__main__":..........
    torch.cuda.empty_cache()
    gc.collect()

    print(f"[*] 실행 장치: {DAiSEEConfig.DEVICE}")
    
    try:
        ds = DAiSEEDataset(DAiSEEConfig.TRAIN_CSV, DAiSEEConfig.VIDEO, transforms.Compose([
            transforms.ToTensor(), 
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]))
        
        loader = DataLoader(ds, batch_size=DAiSEEConfig.BATCH_SIZE, shuffle=True, 
                            num_workers=DAiSEEConfig.NUM_WORKERS, pin_memory=False)
        
        model = DAiSEEModel().to(DAiSEEConfig.DEVICE)
        crit = nn.CrossEntropyLoss(weight=DAiSEEConfig.WEIGHTS).to(DAiSEEConfig.DEVICE)
        opt = optim.AdamW(model.parameters(), lr=DAiSEEConfig.LR)
        
        for e in range(50):
            model.train()
            pbar = tqdm(loader, desc=f"DAiSEE E{e+1}")
            
            # 정확도 계산을 위한 변수 초기화
            running_corrects = 0
            processed_samples = 0
            
            for img, lbl in pbar:
                img, lbl = img.to(DAiSEEConfig.DEVICE), lbl.to(DAiSEEConfig.DEVICE)
                
                opt.zero_grad()
                out = model(img)
                loss = crit(out, lbl)
                loss.backward()
                opt.step()
                
                # 실시간 정확도 계산
                _, preds = torch.max(out, 1)
                running_corrects += torch.sum(preds == lbl.data)
                processed_samples += lbl.size(0)
                current_acc = running_corrects.double() / processed_samples
                
                # 진행률 표시줄 업데이트 (Loss와 Accuracy 함께 표시)
                pbar.set_postfix({
                    'Loss': f'{loss.item():.4f}',
                    'Acc': f'{current_acc.item() * 100:.2f}%'
                })
                
            # 에폭 종료 후 모델 저장
            torch.save(model.state_dict(), DAiSEEConfig.SAVE_PATH)
            
    except Exception as fatal_e:
        print(f"\n[!!!] 치명적 오류 발생: {fatal_e}")

[*] 실행 장치: cuda


DAiSEE E8:  30%|███████████████████▍                                            | 1354/4463 [05:26<12:29,  4.15it/s, Loss=0.0000, Acc=100.00%]


KeyboardInterrupt: 

In [4]:
import pandas as pd

csv = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\Labels\AllLabels.csv"

df = pd.read_csv(csv)

for i in range(20):
    print(df.iloc[i,0])

1100011002.avi
1100011003.avi
1100011004.avi
1100011005.avi
1100011006.avi
1100011007.avi
1100011008.avi
1100011009.avi
1100011010.avi
1100011011.avi
1100011012.avi
1100011013.avi
1100011014.avi
1100011015.avi
1100011016.avi
1100011017.avi
1100011018.avi
1100011019.avi
1100011020.avi
1100011021.avi


In [5]:
import os

root = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation"

cnt = 0
for r,_,files in os.walk(root):
    for f in files:
        if f.endswith(".avi"):
            print(os.path.join(r,f))
            cnt += 1
            if cnt == 20:
                break
    if cnt == 20:
        break

F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation\400022\4000221001\4000221001.avi
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation\400022\4000221002\4000221002.avi
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation\400022\4000221006\4000221006.avi
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation\400022\4000221008\4000221008.avi
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation\400022\4000221009\4000221009.avi
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation\400022\4000221010\4000221010.avi
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation\400022\4000221011\4000221011.avi
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation\400022\4000221013\4000221013.avi
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation\400022\4000221014\4000221014.avi
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_s

In [6]:
VAL_ROOT = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Train"

In [1]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
from torchvision import transforms
from sklearn.metrics import accuracy_score
from torchvision.models.video import swin3d_t

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\test\stable_daisee_model.pth"
VAL_ROOT = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Train"
LABEL_CSV = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\Labels\AllLabels.csv"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class DAiSEEModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.swin = swin3d_t(weights=None)
        self.swin.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.swin.head.in_features, 4)
        )
    def forward(self,x):
        return self.swin(x.transpose(1,2))

model = DAiSEEModel()
state = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state)
model.to(DEVICE)
model.eval()

df = pd.read_csv(LABEL_CSV)

label_map = {}
for _,row in df.iterrows():
    key = str(row.iloc[0]).replace(".avi","")
    label_map[key] = int(row.iloc[2])

def load_video(path):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    if total > 0:
        idxs = np.linspace(0,total-1,16).astype(int)
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES,i)
            ret,f = cap.read()
            if not ret:
                continue
            f = cv2.cvtColor(f,cv2.COLOR_BGR2RGB)
            f = cv2.resize(f,(224,224))
            f = transform(f)
            frames.append(f)
    cap.release()
    if len(frames)==0:
        return None
    while len(frames)<16:
        frames.append(frames[-1])
    return torch.stack(frames)

videos = []
for root,_,files in os.walk(VAL_ROOT):
    for f in files:
        if f.endswith(".avi"):
            videos.append(os.path.join(root,f))

videos = videos[:200]

y_true = []
y_pred = []

for vpath in tqdm(videos):

    name = os.path.basename(vpath).replace(".avi","")

    if name not in label_map:
        continue

    frames = load_video(vpath)
    if frames is None:
        continue

    frames = frames.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(frames)
        pred = torch.argmax(out,1).item()

    y_true.append(label_map[name])
    y_pred.append(pred)

print("samples =", len(y_true))
print("ACC =", accuracy_score(y_true,y_pred))

C:\Users\msi\AppData\Local\Temp\ipykernel_14324\352859759.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(MODEL_PATH, map_location=DEVICE)
100%|█████

samples = 200
ACC = 0.0


In [2]:
import os
import cv2
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import swin3d_t, Swin3D_T_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE"
VIDEO_ROOT = os.path.join(ROOT,"DataSet","Train")
CSV_PATH = os.path.join(ROOT,"Labels","AllLabels.csv")

BATCH = 2
EPOCHS = 20
LR = 1e-4

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

df = pd.read_csv(CSV_PATH)

df["name"] = df.iloc[:,0].apply(lambda x : str(x).replace(".avi",""))
df["label"] = df.iloc[:,2]

train_df , val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

class DaiseeDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def load_video(self,path):

        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames = []

        if total > 0:
            idxs = np.linspace(0,total-1,16).astype(int)

            for i in idxs:
                cap.set(cv2.CAP_PROP_POS_FRAMES,i)
                ret,f = cap.read()

                if not ret:
                    continue

                f = cv2.cvtColor(f,cv2.COLOR_BGR2RGB)
                f = cv2.resize(f,(224,224))
                f = transform(f)
                frames.append(f)

        cap.release()

        if len(frames)==0:
            frames = [torch.zeros(3,224,224)]*16

        while len(frames)<16:
            frames.append(frames[-1])

        return torch.stack(frames)

    def __getitem__(self,idx):

        row = self.df.iloc[idx]

        name = row["name"]
        label = int(row["label"])

        folder1 = name[:6]
        folder2 = name

        path = os.path.join(
            VIDEO_ROOT,
            folder1,
            folder2,
            name + ".avi"
        )

        video = self.load_video(path)

        return video , label

train_ds = DaiseeDataset(train_df)
val_ds = DaiseeDataset(val_df)

train_loader = DataLoader(train_ds,batch_size=BATCH,shuffle=True)
val_loader = DataLoader(val_ds,batch_size=BATCH)

class Model(nn.Module):

    def __init__(self):
        super().__init__()
        self.backbone = swin3d_t(weights=Swin3D_T_Weights.DEFAULT)
        self.backbone.head = nn.Linear(self.backbone.head.in_features,4)

    def forward(self,x):
        return self.backbone(x.transpose(1,2))

model = Model().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(),lr=LR)

for epoch in range(EPOCHS):

    model.train()

    preds_all = []
    labels_all = []

    pbar = tqdm(train_loader)

    for x,y in pbar:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        out = model(x)

        loss = criterion(out,y)

        loss.backward()
        optimizer.step()

        pred = torch.argmax(out,1)

        preds_all.extend(pred.cpu().numpy())
        labels_all.extend(y.cpu().numpy())

        acc = accuracy_score(labels_all,preds_all)

        pbar.set_description(f"train loss {loss.item():.4f} acc {acc:.3f}")

    model.eval()

    v_preds = []
    v_labels = []

    with torch.no_grad():

        for x,y in val_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            out = model(x)

            pred = torch.argmax(out,1)

            v_preds.extend(pred.cpu().numpy())
            v_labels.extend(y.cpu().numpy())

    v_acc = accuracy_score(v_labels,v_preds)

    print(f"VAL ACC = {v_acc}")

    torch.save(model.state_dict(),"daisee_swin_best.pth")

train loss 0.8462 acc 0.478: 100%|████████████████████████████████████████████████████████████████████████| 3570/3570 [26:32<00:00,  2.24it/s]


KeyboardInterrupt: 


KeyboardInterrupt



In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import swin3d_t, Swin3D_T_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE"
VIDEO_ROOT = os.path.join(ROOT, "DataSet", "Train")
CSV_PATH = os.path.join(ROOT, "Labels", "AllLabels.csv")

BATCH = 1
EPOCHS = 20
LR = 1e-4

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

df = pd.read_csv(CSV_PATH)
df["name"] = df.iloc[:,0].astype(str).str.replace(".avi","",regex=False)
df["label"] = df.iloc[:,2].astype(int)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

class DaiseeDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def load_video(self, path):

        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames = []

        if total > 0:
            idxs = np.linspace(0, total-1, 16).astype(int)

            for i in idxs:
                cap.set(cv2.CAP_PROP_POS_FRAMES, i)
                ret, f = cap.read()

                if not ret:
                    continue

                f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
                f = cv2.resize(f, (224,224))
                f = transform(f)
                frames.append(f)

        cap.release()

        if len(frames) == 0:
            frames = [torch.zeros(3,224,224)] * 16

        while len(frames) < 16:
            frames.append(frames[-1])

        return torch.stack(frames)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        name = row["name"]
        label = row["label"]

        folder1 = name[:6]
        folder2 = name

        path = os.path.join(
            VIDEO_ROOT,
            folder1,
            folder2,
            name + ".avi"
        )

        video = self.load_video(path)

        return video, torch.tensor(label, dtype=torch.long)

train_ds = DaiseeDataset(train_df)
val_ds = DaiseeDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH)

class Model(nn.Module):

    def __init__(self):
        super().__init__()
        self.backbone = swin3d_t(weights=Swin3D_T_Weights.DEFAULT)
        self.backbone.head = nn.Linear(self.backbone.head.in_features, 4)

    def forward(self, x):
        return self.backbone(x.transpose(1,2))

model = Model().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)

best_acc = 0

for epoch in range(EPOCHS):

    model.train()

    train_preds = []
    train_labels = []

    pbar = tqdm(train_loader)

    for x, y in pbar:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        pred = torch.argmax(out, 1)

        train_preds.extend(pred.cpu().numpy())
        train_labels.extend(y.cpu().numpy())

        acc = accuracy_score(train_labels, train_preds)

        pbar.set_description(f"E{epoch+1} loss {loss.item():.4f} acc {acc:.3f}")

    model.eval()

    val_preds = []
    val_labels = []

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            out = model(x)
            pred = torch.argmax(out, 1)

            val_preds.extend(pred.cpu().numpy())
            val_labels.extend(y.cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)

    print(f"\nEPOCH {epoch+1} VAL ACC = {val_acc:.4f}")

    torch.save(model.state_dict(), "daisee_last.pth")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "daisee_best.pth")

E1 loss 0.2532 acc 0.583:   1%|▌                                                                            | 48/7140 [00:14<33:02,  3.58it/s]